In [2]:
%pip install numpy pandas nltk

In [3]:
import numpy as np
import pandas as pd
import math
import nltk

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from collections import Counter

nltk.download('wordnet')
nltk.download('omw-1.4')

student_name = "Your Name"
student_id = "Your Student ID"

print("Assignment Beta - Question 4")
print("Student:", student_name)
print("Student ID:", student_id)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Assignment Beta - Question 4
Student: Your Name
Student ID: Your Student ID


In [4]:
classes = ["Positive", "Negative", "Neutral"]

stage1 = np.array([
    [100, 5, 15],
    [8, 80, 12],
    [20, 10, 50]
])

stage2 = np.array([
    [106, 4, 10],
    [6, 84, 10],
    [9, 7, 64]
])

print("Stage 1 Confusion Matrix")
display(pd.DataFrame(stage1, index=classes, columns=classes))

print("Stage 2 Confusion Matrix")
display(pd.DataFrame(stage2, index=classes, columns=classes))

Stage 1 Confusion Matrix


,Positive,Negative,Neutral
Positive,100,5,15
Negative,8,80,12
Neutral,20,10,50


Stage 2 Confusion Matrix


,Positive,Negative,Neutral
Positive,106,4,10
Negative,6,84,10
Neutral,9,7,64


In [5]:
correct = np.trace(stage1)
total = stage1.sum()

accuracy = correct / total

z = 1.96

denominator = 1 + (z**2 / total)

center = (accuracy + z**2 / (2 * total)) / denominator

margin = (z * math.sqrt(
    (accuracy * (1 - accuracy) / total) +
    (z**2 / (4 * total**2))
)) / denominator

lower = center - margin
upper = center + margin

print("Correct Predictions:", correct)
print("Total Reviews:", total)
print(f"Accuracy: {accuracy:.5f} ({accuracy*100:.2f}%)")
print(f"95% Confidence Interval: {lower*100:.2f}% - {upper*100:.2f}%")

Correct Predictions: 230
Total Reviews: 300
Accuracy: 0.76667 (76.67%)
95% Confidence Interval: 71.56% - 81.10%


In [6]:
def calculate_metrics(cm):

    tp = np.diag(cm)
    actual = cm.sum(axis=1)
    predicted = cm.sum(axis=0)

    precision = tp / predicted
    recall = tp / actual

    f1 = 2 * precision * recall / (precision + recall)

    return pd.DataFrame({
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Support": actual
    }, index=classes)


metrics1 = calculate_metrics(stage1)

print("Stage 1 Classification Metrics")
display(metrics1.round(5))

Stage 1 Classification Metrics


,Precision,Recall,F1-score,Support
Positive,0.78125,0.83333,0.80645,120
Negative,0.84211,0.80000,0.82051,100
Neutral,0.64935,0.62500,0.63694,80


In [7]:
macro_f1 = metrics1["F1-score"].mean()

weighted_f1 = np.average(
    metrics1["F1-score"],
    weights=metrics1["Support"]
)

micro_f1 = np.trace(stage1) / stage1.sum()

print(f"Macro F1: {macro_f1:.5f}")
print(f"Weighted F1: {weighted_f1:.5f}")
print(f"Micro F1: {micro_f1:.5f}")

Macro F1: 0.75464
Weighted F1: 0.76594
Micro F1: 0.76667


In [8]:
print("Stage 1 Error Analysis")

for i, actual_class in enumerate(classes):
    errors = stage1[i].sum() - stage1[i, i]

    print(f"\n{actual_class}:")
    print("Incorrect Predictions:", errors)

    for j, predicted_class in enumerate(classes):
        if i != j and stage1[i, j] > 0:
            print(
                f"Actual {actual_class} predicted as "
                f"{predicted_class}: {stage1[i, j]}"
            )

Stage 1 Error Analysis

Positive:
Incorrect Predictions: 20
Actual Positive predicted as Negative: 5
Actual Positive predicted as Neutral: 15

Negative:
Incorrect Predictions: 20
Actual Negative predicted as Positive: 8
Actual Negative predicted as Neutral: 12

Neutral:
Incorrect Predictions: 30
Actual Neutral predicted as Positive: 20
Actual Neutral predicted as Negative: 10


In [9]:
metrics2 = calculate_metrics(stage2)

print("Stage 2 Classification Metrics")
display(metrics2.round(5))

accuracy2 = np.trace(stage2) / stage2.sum()

macro_f1_2 = metrics2["F1-score"].mean()

weighted_f1_2 = np.average(
    metrics2["F1-score"],
    weights=metrics2["Support"]
)

micro_f1_2 = np.trace(stage2) / stage2.sum()

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro F1",
        "Weighted F1",
        "Micro F1",
        "Positive F1",
        "Negative F1",
        "Neutral F1"
    ],

    "Stage 1": [
        accuracy,
        macro_f1,
        weighted_f1,
        micro_f1,
        metrics1.loc["Positive", "F1-score"],
        metrics1.loc["Negative", "F1-score"],
        metrics1.loc["Neutral", "F1-score"]
    ],

    "Stage 2": [
        accuracy2,
        macro_f1_2,
        weighted_f1_2,
        micro_f1_2,
        metrics2.loc["Positive", "F1-score"],
        metrics2.loc["Negative", "F1-score"],
        metrics2.loc["Neutral", "F1-score"]
    ]
})

print("Stage 1 vs Stage 2")
display(comparison.round(5))

Stage 2 Classification Metrics


,Precision,Recall,F1-score,Support
Positive,0.87603,0.88333,0.87967,120
Negative,0.88421,0.84000,0.86154,100
Neutral,0.76190,0.80000,0.78049,80


Stage 1 vs Stage 2


,Metric,Stage 1,Stage 2
0,Accuracy,0.76667,0.84667
1,Macro F1,0.75464,0.84056
2,Weighted F1,0.76594,0.84718
3,Micro F1,0.76667,0.84667
4,Positive F1,0.80645,0.87967
5,Negative F1,0.82051,0.86154
6,Neutral F1,0.63694,0.78049


In [10]:
accuracies = np.array([
    0.847,
    0.840,
    0.853,
    0.837,
    0.850
])

mean_accuracy = np.mean(accuracies)

std_sample = np.std(accuracies, ddof=1)

print(f"Mean Accuracy: {mean_accuracy:.5f}")
print(f"Standard Deviation: {std_sample:.5f}")

print(
    f"Final Result: {mean_accuracy*100:.2f}% "
    f"+/- {std_sample*100:.3f} percentage points"
)

Mean Accuracy: 0.84540
Standard Deviation: 0.00673
Final Result: 84.54% +/- 0.673 percentage points


In [11]:
reference = "the delivery was late but the product quality is good"

candidate = "the product quality is good but delivery was slow"

reference_tokens = reference.split()
candidate_tokens = candidate.split()

# BLEU-4
bleu = sentence_bleu(
    [reference_tokens],
    candidate_tokens,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=SmoothingFunction().method1
)

# ROUGE calculations
def get_ngrams(tokens, n):
    return Counter(
        tuple(tokens[i:i+n])
        for i in range(len(tokens)-n+1)
    )

def rouge_score(reference, candidate, n):

    ref_ngrams = get_ngrams(reference, n)
    cand_ngrams = get_ngrams(candidate, n)

    overlap = sum((ref_ngrams & cand_ngrams).values())

    precision = overlap / sum(cand_ngrams.values())
    recall = overlap / sum(ref_ngrams.values())

    f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


rouge1 = rouge_score(reference_tokens, candidate_tokens, 1)
rouge2 = rouge_score(reference_tokens, candidate_tokens, 2)

# ROUGE-L using Longest Common Subsequence
def lcs_length(x, y):

    dp = [[0] * (len(y)+1) for _ in range(len(x)+1)]

    for i in range(1, len(x)+1):
        for j in range(1, len(y)+1):

            if x[i-1] == y[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1

            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    return dp[-1][-1]


lcs = lcs_length(reference_tokens, candidate_tokens)

rougeL_precision = lcs / len(candidate_tokens)
rougeL_recall = lcs / len(reference_tokens)

rougeL_f1 = (
    2 * rougeL_precision * rougeL_recall /
    (rougeL_precision + rougeL_recall)
)

# METEOR
meteor = meteor_score(
    [reference_tokens],
    candidate_tokens
)

results = pd.DataFrame({
    "Metric": [
        "BLEU-4",
        "ROUGE-1 Precision",
        "ROUGE-1 Recall",
        "ROUGE-1 F1",
        "ROUGE-2 Precision",
        "ROUGE-2 Recall",
        "ROUGE-2 F1",
        "ROUGE-L Precision",
        "ROUGE-L Recall",
        "ROUGE-L F1",
        "METEOR"
    ],

    "Score": [
        bleu,
        rouge1[0], rouge1[1], rouge1[2],
        rouge2[0], rouge2[1], rouge2[2],
        rougeL_precision, rougeL_recall, rougeL_f1,
        meteor
    ]
})

display(results.round(5))

,Metric,Score
0,BLEU-4,0.47495
1,ROUGE-1 Precision,0.88889
2,ROUGE-1 Recall,0.80000
3,ROUGE-1 F1,0.84211
4,ROUGE-2 Precision,0.62500
5,ROUGE-2 Recall,0.55556
6,ROUGE-2 F1,0.58824
7,ROUGE-L Precision,0.55556
8,ROUGE-L Recall,0.50000
9,ROUGE-L F1,0.52632


In [12]:
probabilities = np.array([
    0.60, 0.40, 0.50,
    0.80, 0.70, 0.30,
    0.25, 0.50, 0.20
])

N = len(probabilities)

perplexity = np.exp(
    -np.mean(np.log(probabilities))
)

print("Number of Tokens:", N)
print(f"Perplexity: {perplexity:.5f}")

Number of Tokens: 9
Perplexity: 2.32486


In [13]:
final_results = pd.DataFrame({
    "Question": [
        "Q1 Accuracy",
        "Q1 95% CI Lower",
        "Q1 95% CI Upper",
        "Q3 Macro F1",
        "Q3 Weighted F1",
        "Q3 Micro F1",
        "Q5 Stage 2 Accuracy",
        "Q6 Mean Accuracy",
        "Q6 Standard Deviation",
        "Q8 Perplexity"
    ],

    "Result": [
        accuracy,
        lower,
        upper,
        macro_f1,
        weighted_f1,
        micro_f1,
        accuracy2,
        mean_accuracy,
        std_sample,
        perplexity
    ]
})

display(final_results.round(5))

,Question,Result
0,Q1 Accuracy,0.76667
1,Q1 95% CI Lower,0.71562
2,Q1 95% CI Upper,0.81097
3,Q3 Macro F1,0.75464
4,Q3 Weighted F1,0.76594
5,Q3 Micro F1,0.76667
6,Q5 Stage 2 Accuracy,0.84667
7,Q6 Mean Accuracy,0.84540
8,Q6 Standard Deviation,0.00673
9,Q8 Perplexity,2.32486
